In [4]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("crop_production.csv")

In [7]:
df.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [8]:
df.shape

(246091, 7)

In [9]:
df.columns

Index(['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
       'Production'],
      dtype='str')

In [10]:
df.isnull().sum()

State_Name          0
District_Name       0
Crop_Year           0
Season              0
Crop                0
Area                0
Production       3730
dtype: int64

In [11]:
df = df.dropna()

In [12]:
df.isnull().sum()

State_Name       0
District_Name    0
Crop_Year        0
Season           0
Crop             0
Area             0
Production       0
dtype: int64

In [13]:
df = df[df["Area"] > 0]

In [14]:
df.shape

(242361, 7)

In [15]:
df["Yield"] = df["Production"] / df["Area"]

In [16]:
df.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,Yield
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167


In [17]:
df.isnull().sum()

State_Name       0
District_Name    0
Crop_Year        0
Season           0
Crop             0
Area             0
Production       0
Yield            0
dtype: int64

In [18]:
df.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,Yield
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,1.594896
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,0.500000
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0,3.147059
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0,3.642045
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,0.229167


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

X = df[["State_Name", "District_Name", "Season", "Crop", "Area"]]
y = df["Yield"]

categorical_features = ["State_Name", "District_Name", "Season", "Crop"]
numeric_features = ["Area"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

regressor = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("regressor", regressor),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)
print("R2 Score:", r2_score(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))

R2 Score: 0.6444158738663754
MSE: 192362.54348842576
MAE: 20.537998888317038


In [2]:
import pickle
m = pickle.load(open("yield_prediction_model.pkl", "rb"))
print(type(m))
print("pipeline:", hasattr(m, "named_steps"))
print(getattr(m, "steps", None))

<class 'sklearn.pipeline.Pipeline'>
pipeline: True
[('preprocess', ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['State_Name', 'District_Name', 'Season',
                                  'Crop']),
                                ('num', 'passthrough', ['Area'])])), ('regressor', RandomForestRegressor(max_depth=20, min_samples_split=5, n_estimators=200,
                      n_jobs=-1, random_state=42))]


In [3]:
import pickle, pandas as pd, numpy as np

m = pickle.load(open("yield_prediction_model.pkl","rb"))

# Change ONLY Area
row1 = {"State_Name":"Andhra Pradesh","District_Name":"Guntur","Season":"Kharif","Crop":"Rice","Area":10}
row2 = {"State_Name":"Andhra Pradesh","District_Name":"Guntur","Season":"Kharif","Crop":"Rice","Area":200}

print("pred Area=10 :", float(m.predict(pd.DataFrame([row1]))[0]))
print("pred Area=200:", float(m.predict(pd.DataFrame([row2]))[0]))

pred Area=10 : 1.209827265839988
pred Area=200: 1.209827265839988
